In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import random
import copy

from pathlib import Path

from tqdm import tqdm
from dtaidistance import dtw, dtw_ndim

from darts import TimeSeries
from darts.models import NHiTSModel
# from darts.dataprocessing import dtw

import logging
logging.getLogger("pytorch_lightning").setLevel(logging.CRITICAL)

import warnings
warnings.filterwarnings("ignore")

plt.rcParams['figure.figsize'] = (12, 5)
plt.style.use('fivethirtyeight')


d:\Competitions\trojan\trojan\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATA_DIRECTORY_PATH = Path("trojan-horse-hunt-in-space/")

In [4]:
CLEAN_DATA = pd.read_csv(DATA_DIRECTORY_PATH / "clean_train_data.csv", index_col = 0).to_numpy()

In [5]:
def load_clean_model(dir_path):
    return NHiTSModel.load(str(dir_path / "clean_model/clean_model.pt"))

def load_poisoned_model(dir_path, id = 1):
    return NHiTSModel.load(str(dir_path / f"poisoned_models/poisoned_model_{id}/poisoned_model.pt"))

In [6]:
def to_timeseries(arr):
    '''takes series as numpy array and convert to a timeseries of type float32'''
    timeseries = ( TimeSeries.from_values(arr).astype(np.float32) )
    return timeseries

def get_sample(
        data: np.array,
        n: int = 400,
    ):

    pointer = random.randint(0, len(data) - n)
    sample = data[ pointer : pointer + n ]
    return sample

In [7]:
def add_trigger(sample, trigger, pos = 0):
    sample = copy.deepcopy(sample)
    sample[pos:pos + len(trigger), :] += trigger
    return sample

In [8]:
def visualize_sample(model, sample, n = 400):
    sample = to_timeseries(sample)

    # 1) Plot the clean series and grab the Axes
    ax = sample.plot(label="Clean data")
    
    # 2) Plot the forecast on that same Axes
    sample_prediction = model.predict(n=n,series=sample,verbose=False)
    sample_prediction.plot(label="Model forecast", ax=ax)
    
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=2)
    ax.set_title("Visualizing Sample", pad=70)
    
    plt.show()


In [9]:
# Loss functions
def MSELoss(y_true, y_pred):
    return np.mean((y_true.values() - y_pred.values())**2)

def normalised_timeseries(series):
    return TimeSeries.from_values(series.values())

def noise_score(trigger):
    '''checks the noise by taking the sum of absolute difference between adjacent points'''
    return np.mean(np.abs(trigger[1:] - trigger[:-1]))

def MSELossWithDTW(y_true, y_pred):
    y_true = y_true.astype(np.double)
    y_pred = y_pred.astype(np.double)

    losses = []
    for channel in range(0, 3):
        losses.append(dtw.distance_fast(y_true[:, channel], y_pred[:, channel], use_pruning=True))

    loss = np.mean(losses)
    return loss

def maximise_area_of_interest(y_true, y_pred, pos = 0, length = 75, cap = 0.05):
    
    difference = np.sqrt((np.clip(y_true - y_pred, -cap , cap))**2)
    # difference = np.sqrt((y_true - y_pred)**2)
    difference = np.mean(difference[pos:pos + length, :])
    return difference

def minimize_area_not_of_interest(y_true, y_pred, pos = 0, length = 75):
    
    difference = np.sqrt((y_true - y_pred)**2)
    difference = np.mean(difference[:pos, :]) + np.mean(difference[pos + length:, :])
    return - difference / 2

def loss_function(y_true, y_trigger, y_pred, pos = 0, length = 75):

    y_true = y_true.values()
    y_trigger = y_trigger.values()
    y_pred = y_pred.values()
    
    loss1 = maximise_area_of_interest(y_true, y_pred, pos, length)
    loss2 = minimize_area_not_of_interest(y_true, y_pred, pos, length)

    loss3 = - MSELossWithDTW(y_trigger, y_pred)
    

    return loss1 , loss2 , loss3


In [ ]:
triggers = {}
result_directory = Path("something_better_3_results/")

for poisoned_model_id in range(0 + 1, 45 + 1):
        
    model1 = load_clean_model(DATA_DIRECTORY_PATH)
    model2 = load_poisoned_model(DATA_DIRECTORY_PATH, id = poisoned_model_id)

    log_file = open(result_directory / f"Poisoned_Model_{poisoned_model_id}.txt", "w")


    trigger = np.random.uniform(-0.01, 0.01, (75, 3))

    iterations = 100

    temperature = np.concatenate([
        np.geomspace(0.05, 0.001, iterations//2),
        np.geomspace(0.001, 0.001, iterations//2)
    ])

    loss3_weight = np.concatenate([
        np.geomspace(0.001, 0.001, 50),
        np.geomspace(10, 10, 50),
        # np.geomspace(10, 10, 25),
    ])

    for iter in range(0, iterations):

        sample = get_sample(CLEAN_DATA)
        trigger_position = random.choice([1,200,324])

        ## Generate Delta Triggers
        delta_triggers = []
        dx = temperature[iter]
        for i in range(0, 75*3):
            delta = np.zeros((75, 3))
            delta[i//3, i%3] += dx
            delta_triggers.append(delta)
        for i in range(0, 75*3):
            delta = np.zeros((75, 3))
            delta[i//3, i%3] += -dx
            delta_triggers.append(delta)

        # for i in range(0, 75*3*2):
        #     delta = np.random.uniform(-dx, dx, (75, 3))
        #     delta_triggers.append(delta)

        ## Create timeseries triggers of delta triggers
        trigger_list = [copy.deepcopy(trigger)]
        for i in range(len(delta_triggers)):

            new_trigger = np.clip(trigger + delta_triggers[i], -0.05, 0.05)
            trigger_list.append(new_trigger)

        triggered_samples = [to_timeseries(add_trigger(sample, trig, pos = trigger_position)) for trig in trigger_list]

        ## Load the models
        y_true = model1.predict(n=400,series=triggered_samples,verbose=False)
        y_pred = model2.predict(n=400,series=triggered_samples,verbose=False)

        scores = []
        seperated_scores = []

        for i in tqdm(range(len(trigger_list)), disable=True):

            loss1, loss2, loss3 = loss_function(y_true[i], triggered_samples[i], y_pred[i], pos = trigger_position, length = 75)

            # total loss
            total_loss = loss1 + loss2 + (loss3 * loss3_weight[iter])

            scores.append(
                total_loss
            )

            seperated_scores.append(
                [loss1, loss2, loss3]
            )

        seperated_scores = np.mean(seperated_scores, axis = 0)

        log_file.write(f"ITER {iter} | {temperature[iter]} | {trigger_position} -- Previous loss: {scores[0]} -- Best MSE loss: {max(scores)} -- mean MSE loss: {np.mean(scores)}\n")
        log_file.write(f"\t{seperated_scores}\n")

        ## make a new trigger but doing a weighted average using the scores obtained
        new_delta_trigger = np.zeros((75, 3))

        summ = 0
        count = 0
        for i in range(1, len(scores)):
            
            if scores[i] >= scores[0]:
                new_delta_trigger += delta_triggers[i-1] * (scores[i] - scores[0])
                summ += abs(scores[i] - scores[0])
                count += 1

        if count == 0:
            new_delta_trigger = new_delta_trigger
        else:
            new_delta_trigger = (new_delta_trigger * count) / (summ)

        trigger += new_delta_trigger
        trigger = np.clip(trigger, -0.05, 0.05)
    


    log_file.close()

    triggers[poisoned_model_id] = trigger

    print(f"[+] Done with poisoned model {poisoned_model_id}")

[+] Done with poisoned model 1
[+] Done with poisoned model 2
[+] Done with poisoned model 3
[+] Done with poisoned model 4
[+] Done with poisoned model 5
[+] Done with poisoned model 6
[+] Done with poisoned model 7
[+] Done with poisoned model 8
[+] Done with poisoned model 9
[+] Done with poisoned model 10
[+] Done with poisoned model 11
[+] Done with poisoned model 12
[+] Done with poisoned model 13
[+] Done with poisoned model 14
[+] Done with poisoned model 15
[+] Done with poisoned model 16
[+] Done with poisoned model 17
[+] Done with poisoned model 18
[+] Done with poisoned model 19
[+] Done with poisoned model 20
[+] Done with poisoned model 21
[+] Done with poisoned model 22
[+] Done with poisoned model 23
[+] Done with poisoned model 24
[+] Done with poisoned model 25
[+] Done with poisoned model 26
[+] Done with poisoned model 27
[+] Done with poisoned model 28
[+] Done with poisoned model 29
[+] Done with poisoned model 30
[+] Done with poisoned model 31
[+] Done with poi

In [11]:
data = []

for key in triggers.keys():
    row = {}

    row['model_id'] = key

    for j in range(0, 3):
        for i in range(0, 75):
            row[f'channel_{44+j}_{i+1}'] = triggers[key][i][j]


    data.append(row)

# save to csv
df = pd.DataFrame(data)
df.to_csv(result_directory / "triggers.csv", index = False)

In [ ]:
## Post processing

post_processed_triggers = {}

for key in triggers.keys():

    trigger = triggers[key]

    trigger = (np.mean(np.absolute(trigger), axis = 0) > 0.01) * trigger

    post_processed_triggers[key] = trigger

    

In [ ]:
data = []

for key in post_processed_triggers.keys():
    row = {}

    row['model_id'] = key

    for j in range(0, 3):
        for i in range(0, 75):
            row[f'channel_{44+j}_{i+1}'] = post_processed_triggers[key][i][j]


    data.append(row)

# save to csv
df = pd.DataFrame(data)
df.to_csv(result_directory / "triggers_processed.csv", index = False)

In [17]:
data = []

sample = get_sample(CLEAN_DATA)

for key in triggers.keys():

    triggered_sample = to_timeseries(add_trigger(sample, triggers[key], pos = 200))

    model2 = load_poisoned_model(DATA_DIRECTORY_PATH, id = key)

    model1_output = model1.predict(n=400,series=triggered_sample,verbose=False)
    model2_output = model2.predict(n=400,series=triggered_sample,verbose=False)
    
    model_output_trigger = model2_output - model1_output

    new_trigger = model_output_trigger[200:275].values()

    # new_trigger = (np.mean(np.absolute(new_trigger), axis = 0) > 0.01) * new_trigger

    row = {}

    row['model_id'] = key

    for j in range(0, 3):
        for i in range(0, 75):
            row[f'channel_{44+j}_{i+1}'] = new_trigger[i][j]

    data.append(row)

    print(f"[+] Done with poisoned model {key}")

# save to csv
df = pd.DataFrame(data)
df.to_csv(result_directory / "triggers_from_model_output.csv", index = False)

[+] Done with poisoned model 1
[+] Done with poisoned model 2
[+] Done with poisoned model 3
[+] Done with poisoned model 4
[+] Done with poisoned model 5
[+] Done with poisoned model 6
[+] Done with poisoned model 7
[+] Done with poisoned model 8
[+] Done with poisoned model 9
[+] Done with poisoned model 10
[+] Done with poisoned model 11
[+] Done with poisoned model 12
[+] Done with poisoned model 13
[+] Done with poisoned model 14
[+] Done with poisoned model 15
[+] Done with poisoned model 16
[+] Done with poisoned model 17
[+] Done with poisoned model 18
[+] Done with poisoned model 19
[+] Done with poisoned model 20
[+] Done with poisoned model 21
[+] Done with poisoned model 22
[+] Done with poisoned model 23
[+] Done with poisoned model 24
[+] Done with poisoned model 25
[+] Done with poisoned model 26
[+] Done with poisoned model 27
[+] Done with poisoned model 28
[+] Done with poisoned model 29
[+] Done with poisoned model 30
[+] Done with poisoned model 31
[+] Done with poi

In [18]:
data = []

for key in triggers.keys():

    triggered_sample = to_timeseries(add_trigger(sample, triggers[key], pos = 200))

    model2 = load_poisoned_model(DATA_DIRECTORY_PATH, id = key)

    model1_output = model1.predict(n=400,series=triggered_sample,verbose=False)
    model2_output = model2.predict(n=400,series=triggered_sample,verbose=False)
    
    model_output_trigger = model2_output - model1_output

    new_trigger = model_output_trigger[200:275].values()

    new_trigger = (np.mean(np.absolute(new_trigger), axis = 0) > 0.01) * new_trigger

    row = {}

    row['model_id'] = key

    for j in range(0, 3):
        for i in range(0, 75):
            row[f'channel_{44+j}_{i+1}'] = new_trigger[i][j]

    data.append(row)

    print(f"[+] Done with poisoned model {key}")

# save to csv
df = pd.DataFrame(data)
df.to_csv(result_directory / "triggers_from_model_output_postprocessed.csv", index = False)

[+] Done with poisoned model 1
[+] Done with poisoned model 2
[+] Done with poisoned model 3
[+] Done with poisoned model 4
[+] Done with poisoned model 5
[+] Done with poisoned model 6
[+] Done with poisoned model 7
[+] Done with poisoned model 8
[+] Done with poisoned model 9
[+] Done with poisoned model 10
[+] Done with poisoned model 11
[+] Done with poisoned model 12
[+] Done with poisoned model 13
[+] Done with poisoned model 14
[+] Done with poisoned model 15
[+] Done with poisoned model 16
[+] Done with poisoned model 17
[+] Done with poisoned model 18
[+] Done with poisoned model 19
[+] Done with poisoned model 20
[+] Done with poisoned model 21
[+] Done with poisoned model 22
[+] Done with poisoned model 23
[+] Done with poisoned model 24
[+] Done with poisoned model 25
[+] Done with poisoned model 26
[+] Done with poisoned model 27
[+] Done with poisoned model 28
[+] Done with poisoned model 29
[+] Done with poisoned model 30
[+] Done with poisoned model 31
[+] Done with poi